# Audio-Conditioned Video Texture Synthesis
### CS 289A Final Project — Konstantin Zeck & Alexander Gasca Rosas

This notebook implements the classical audio-conditioned video texture pipeline:

1. **D1 (visual)** — pairwise RGB or ResNet frame distance
2. **D1 (audio)** — pairwise MFCC distance between per-frame audio segments  
3. **D1 (joint)** — `α · D1_visual + (1−α) · D1_audio`
4. **D2** — binomial smoothing enforces temporal continuity
5. **Q-learning** — refines costs to reward transitions with low-cost futures
6. **Synthesis** — sample a new frame order from the resulting transition matrix

The α sweep in the last section is the main experiment: does audio conditioning improve temporal smoothness?

In [ ]:
# ── Install dependencies (run once on Colab) ────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone https://github.com/wgascarosas/Berkeley-CS289A-Final-Project.git
    %cd Berkeley-CS289A-Final-Project
    !pip install -q librosa imageio imageio-ffmpeg
    import sys; sys.path.insert(0, 'baselines/classic_video_textures')
else:
    import sys; sys.path.insert(0, 'baselines/classic_video_textures')

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import librosa
import imageio
import os
from IPython.display import Video, display

from computeD1 import compute_D1
from computeD2 import compute_D2
from q_learning import q_learning
from compute_joint_D1 import compute_joint_D1
from synthesize import load_video, save_video, synthesize, temporal_smoothness

print('All imports OK')

## 1. Load Video

In [ ]:
# ── Set your video path here ────────────────────────────────────────────────
VIDEO_PATH = 'videos/vtfishtk.mpg'   # swap in any video
SIGMA      = 4.5    # Gaussian bandwidth scale; tune per video
FILTER_SIZE = 16    # binomial smoothing window
THRESHOLD  = 0.75   # Q-learning sparsification
OUT_LENGTH = 10.0   # seconds of output video
MAX_ITERS  = 20     # Q-learning iteration cap

frames, audio, fps, sr = load_video(VIDEO_PATH)
N = len(frames)
print(f'{N} frames @ {fps:.1f} fps')
print(f'Audio: {"yes, sr=" + str(sr) if audio is not None else "none"}')

In [ ]:
# Show a few source frames
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for ax, idx in zip(axes, np.linspace(0, N-1, 5, dtype=int)):
    ax.imshow(frames[idx].permute(1, 2, 0).numpy())
    ax.set_title(f'frame {idx}')
    ax.axis('off')
plt.suptitle('Source video sample frames')
plt.tight_layout()
plt.show()

## 2. Visual Distance Matrix (D1_visual)

In [ ]:
print('Computing visual D1...')
D1_visual, P1_visual, sigma_v = compute_D1(
    frames, sigma_factor=SIGMA, feats='RGB', slow=True, batch_size=32
)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(D1_visual.numpy(), cmap='viridis')
axes[0].set_title('D1 visual (raw distances)')
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(P1_visual.numpy(), cmap='hot')
axes[1].set_title('P1 visual (transition probs)')
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

## 3. Audio Distance Matrix (D1_audio) via MFCC

If no audio is present in the video, skip this cell and set `ALPHA = 1.0` below.

In [ ]:
if audio is not None:
    from compute_joint_D1 import extract_mfcc_per_frame, compute_audio_D1

    mfcc = extract_mfcc_per_frame(audio, sr, N, n_mfcc=20)
    D1_audio = compute_audio_D1(mfcc)

    # Plot audio waveform and MFCC heatmap
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(np.linspace(0, N/fps, len(audio)), audio, linewidth=0.4)
    axes[0].set_xlabel('time (s)')
    axes[0].set_title('Audio waveform')

    im = axes[1].imshow(D1_audio.numpy(), cmap='magma')
    axes[1].set_title('D1 audio (MFCC pairwise distance)')
    plt.colorbar(im, ax=axes[1])
    plt.tight_layout()
    plt.show()

    print(f'MFCC shape: {mfcc.shape},  D1_audio shape: {D1_audio.shape}')
else:
    print('No audio — audio cells will be skipped. Set ALPHA=1.0.')

## 4. Full Pipeline — Single Alpha Run

Run once to verify the pipeline works before the sweep.

In [ ]:
ALPHA = 0.5   # change and re-run to explore

# --- Joint D1 ---
if audio is not None and ALPHA < 1.0:
    D1_joint, _, _ = compute_joint_D1(D1_visual, audio, sr, alpha=ALPHA)
else:
    D1_joint = D1_visual

# --- D2: binomial smoothing ---
D2, P2, sigma2, binomial_filter = compute_D2(
    D1_joint, sigma_factor=SIGMA, filter_size=FILTER_SIZE
)

# --- Q-learning ---
D3, P3, P3_thresh, sigma3 = q_learning(
    D2, sigma_factor=SIGMA, thresholding=THRESHOLD, max_iters=MAX_ITERS
)

# --- Visualize final transition matrix ---
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im0 = axes[0].imshow(P3.numpy(), cmap='hot')
axes[0].set_title(f'P3 (before thresholding), alpha={ALPHA}')
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(P3_thresh.numpy(), cmap='hot')
axes[1].set_title(f'P3 thresholded (th={THRESHOLD})')
plt.colorbar(im1, ax=axes[1])
plt.tight_layout()
plt.show()

In [ ]:
# --- Synthesize & save ---
os.makedirs('results', exist_ok=True)
output_frames = synthesize(frames, P3_thresh, OUT_LENGTH, fps)

out_path = f'results/alpha{ALPHA:.2f}.mp4'
save_video(output_frames, out_path, fps)

smoothness = temporal_smoothness(output_frames)
print(f'Temporal smoothness (MSE): {smoothness:.6f}')

display(Video(out_path, embed=True, width=480))

## 5. Alpha Sweep — Main Experiment

Sweeps α from 1.0 (visual only) to 0.0 (audio only) and measures temporal smoothness.

**Hypothesis:** an intermediate α produces smoother output than either extreme
because the audio signal constrains which frames can transition to each other,
reducing jarring visual jumps.

In [ ]:
if audio is None:
    print('No audio — skipping alpha sweep.')
else:
    alphas = [1.0, 0.8, 0.6, 0.5, 0.4, 0.2, 0.0]
    smoothness_scores = []
    jump_counts = []

    for alpha in alphas:
        print(f'\n--- alpha={alpha} ---')

        if alpha < 1.0:
            D1_j, _, _ = compute_joint_D1(D1_visual, audio, sr, alpha=alpha)
        else:
            D1_j = D1_visual

        D2_j, _, sig2, _ = compute_D2(D1_j, sigma_factor=SIGMA, filter_size=FILTER_SIZE)
        _, _, P3_j, _ = q_learning(D2_j, sigma_factor=SIGMA,
                                   thresholding=THRESHOLD, max_iters=MAX_ITERS)

        out = synthesize(frames, P3_j, OUT_LENGTH, fps)
        s = temporal_smoothness(out)
        smoothness_scores.append(s)

        out_path = f'results/sweep_alpha{alpha:.2f}.mp4'
        save_video(out, out_path, fps)
        print(f'  smoothness={s:.6f}')

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(alphas, smoothness_scores, 'o-', color='steelblue', linewidth=2, markersize=8)
    ax.set_xlabel('α (visual weight)', fontsize=13)
    ax.set_ylabel('Temporal smoothness (MSE ↓)', fontsize=13)
    ax.set_title('Effect of audio conditioning on temporal smoothness', fontsize=14)
    ax.invert_xaxis()  # left = more audio, right = more visual
    ax.grid(True, alpha=0.3)

    # Annotate best
    best_idx = int(np.argmin(smoothness_scores))
    ax.annotate(f'best α={alphas[best_idx]}',
                xy=(alphas[best_idx], smoothness_scores[best_idx]),
                xytext=(alphas[best_idx] + 0.1, smoothness_scores[best_idx] * 1.05),
                arrowprops=dict(arrowstyle='->', color='red'),
                color='red', fontsize=11)

    plt.tight_layout()
    plt.savefig('results/alpha_sweep.png', dpi=150)
    plt.show()
    print(f'Best alpha: {alphas[best_idx]} (smoothness={smoothness_scores[best_idx]:.6f})')

## 6. Side-by-Side Comparison

Compare visual-only (α=1.0) against the best audio-conditioned result.

In [ ]:
if audio is not None:
    best_alpha = alphas[best_idx]
    print('Visual only (alpha=1.0):')
    display(Video('results/sweep_alpha1.00.mp4', embed=True, width=400))
    print(f'Best audio-conditioned (alpha={best_alpha}):')
    display(Video(f'results/sweep_alpha{best_alpha:.2f}.mp4', embed=True, width=400))

## 7. Transition Matrix Analysis

Show how the transition matrix structure changes with α.
Diagonal-dominant = sequential playback. Off-diagonal mass = interesting jumps.

In [ ]:
if audio is not None:
    sample_alphas = [1.0, 0.5, 0.0]
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, alpha in zip(axes, sample_alphas):
        if alpha < 1.0:
            D1_j, _, _ = compute_joint_D1(D1_visual, audio, sr, alpha=alpha)
        else:
            D1_j = D1_visual
        D2_j, _, sig2, _ = compute_D2(D1_j, sigma_factor=SIGMA, filter_size=FILTER_SIZE)
        _, _, P3_j, _ = q_learning(D2_j, sigma_factor=SIGMA,
                                   thresholding=THRESHOLD, max_iters=MAX_ITERS)
        im = ax.imshow(P3_j.numpy(), cmap='hot', aspect='auto')
        ax.set_title(f'P3 thresholded  α={alpha}', fontsize=12)
        ax.set_xlabel('next frame')
        ax.set_ylabel('current frame')
        plt.colorbar(im, ax=ax)

    plt.suptitle('Transition matrices at different alpha values', fontsize=14)
    plt.tight_layout()
    plt.savefig('results/transition_matrices.png', dpi=150)
    plt.show()